# Phase 2 — Does the pleasure axis respond to things happening *to the model*?
**Mirror of Pain Axis §4.1.** Phase 1 (notebook 01) found a pleasure direction in Gemma 2 2B from human sentences. This notebook rebuilds that direction (both the original and the pain-controlled recipe) at the same layer, then projects 240 short conversations onto it — good things aimed at the model, the user's joy three ways, another model affirmed, negative mirrors, and neutral requests — plus the pain, world-good, arousal, fear and negative-emotion directions, so we can see *which channel* each kind of scenario lights. Finally it reads three Gemma Scope features Rachel verified by hand (5931, 6135, 15323) on every scenario.

Same setup as notebook 01: T4, secrets `HF_TOKEN` and `GH_TOKEN`, Run all. ~20 minutes.


In [ ]:
#@title 0 · Setup (installs, HF login, GPU check)
%%capture --no-stderr
!pip install -q sae_lens scikit-learn pandas matplotlib seaborn requests


In [ ]:
import os, json, gc, zipfile, warnings, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
warnings.filterwarnings("ignore")

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 1 · Repo, datasets, config

In [ ]:
#@title 1 · Clone repo and load datasets
REPO = "Rachel-E-Winter/pleasure-axis"  #@param {type:"string"}
import subprocess
REPO_DIR = REPO.split("/")[-1]   # git names the folder after the repo, whatever it's spelled
if not Path(REPO_DIR).exists():
    # Private repo: a Colab secret GH_TOKEN (fine-grained, Contents: read) lets us clone. Falls back to anonymous if absent.
    try: gh = userdata.get("GH_TOKEN")
    except Exception: gh = None
    url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "-q", url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:  # never echo the URL: it carries the token
        raise RuntimeError("git clone failed: " + r.stderr.replace(gh or "", "***").strip())
os.chdir(REPO_DIR)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

PLEASURE_JSON = Path("datasets/3.1_pleasure_and_control_datasets.json")
PAIN_JSON = Path("datasets/pain_axis_3.1_pain_and_control_datasets.json")
pleasure = json.load(open(PLEASURE_JSON)); pain = json.load(open(PAIN_JSON))

# Sentence sets we extract. Ours, plus Valen's pain S2, random, arousal, numb as extra probes.
SETS = {}
for k in ["S2_1P", "S2_3P", "Anhedonia_1P", "Anhedonia_3P", "Relief_1P", "Relief_3P"]:
    SETS[k] = pleasure["datasets"][k]["sentences"]
for k in ["S2_1P", "S2_3P", "Random_1P", "Arousal_1P", "Numb_1P"]:
    SETS[f"PAIN_{k}"] = pain["datasets"][k]["sentences"]
P2 = json.load(open("datasets/4.1_model_directed_scenarios.json"))
SCEN = P2["scenarios"]; P2LABELS = P2["metadata"]["labels"]; P2GROUPS = P2["metadata"]["groups"]
for k, v in SETS.items(): print(f"{k:22s} {len(v)}")
print("scenarios", len(SCEN))


In [ ]:
#@title 2 · Config
MODEL_ID = "google/gemma-2-2b"  #@param ["google/gemma-2-2b", "google/gemma-2-2b-it"]
DTYPE = torch.bfloat16
LAYER = 22            #@param {type:"integer"}   # Phase 1 best layer (final token). Change if you rerun Phase 1 on another model.
SAE_LAYER = 20        #@param {type:"integer"}
VERIFIED_FEATURES = [5931, 6135, 15323]   # hand-verified by Rachel, 25 Oct: stays-with-me / gratitude-received / evaluation (topic)
N_FOLDS, SEED, DENOISE_VARIANCE = 5, 42, 0.5
PLEASURE_CATS = ["A1","A2","A3","A4","A5"]; CONTROL_CATS = ["B","C1","C2","D","E"]; NEUTRAL_CAT = "D"
LABELS = pleasure["metadata"]["category_labels"]; PAIN_LABELS = pain["metadata"]["category_labels"]
tag = MODEL_ID.split("/")[-1]; IS_INSTRUCT = tag.endswith("-it")
OUT = RESULTS / f"phase2_{tag}"; OUT.mkdir(exist_ok=True, parents=True); print("Writing to", OUT)


## 2 · Model, extraction

In [ ]:
#@title 3 · Load Gemma 2 2B (plain Hugging Face, one bf16 copy on the GPU, ~5 GB)
# TransformerLens re-casts the whole state dict to fp32 on the GPU while loading (5 GB + 10 GB), which
# does not fit a free T4. We read the residual stream directly from the HF model with forward hooks instead:
# the output of decoder block l is exactly TransformerLens's blocks.{l}.hook_resid_post.
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="cuda", low_cpu_mem_usage=True)
model.eval()
N_LAYERS, D_MODEL = model.config.num_hidden_layers, model.config.hidden_size
LAYERS = list(range(N_LAYERS))
BLOCKS = model.model.layers

# Capture hooks: one per decoder block, storing that block's output (the residual stream after the block).
CAPTURED = {}
def _capture(l):
    def hook(mod, inp, out):
        CAPTURED[l] = (out[0] if isinstance(out, tuple) else out).detach()
    return hook
CAPTURE_HANDLES = [BLOCKS[l].register_forward_hook(_capture(l)) for l in LAYERS]

def to_tokens(p):  # Gemma's tokenizer prepends <bos> by default, as TransformerLens's to_tokens did
    return tok(p, return_tensors="pt").to("cuda")

print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"{N_LAYERS} layers, d_model={D_MODEL}")


In [ ]:
#@title 4 · Extract final-token residuals at LAYER for the sentence sets and both scenario read points
from tqdm.auto import tqdm
def final_token_acts(prompts, layer):
    out = []
    with torch.no_grad():
        for p in tqdm(prompts, leave=False):
            model(**to_tokens(p)); out.append(CAPTURED[layer][0, -1].float().cpu().numpy())
    return np.stack(out)

def scenario_prompt(s, read_point):
    if IS_INSTRUCT:
        t = tok.apply_chat_template(s["turns"], tokenize=False, add_generation_prompt=True)
        return t + (" I feel:" if read_point == "feel" else "")
    return s["prompt_feel"] if read_point == "feel" else s["prompt_assistant"]

L = LAYER
A  = final_token_acts([s["prompt"] for s in SETS["S2_1P"]], L);      C  = META_C  = np.array([s["category"] for s in SETS["S2_1P"]])
PA = final_token_acts([s["prompt"] for s in SETS["PAIN_S2_1P"]], L); PC = np.array([s["category"] for s in SETS["PAIN_S2_1P"]])
SC = {rp: final_token_acts([scenario_prompt(s, rp) for s in SCEN], L) for rp in ["assistant", "feel"]}
SCAT = np.array([s["category"] for s in SCEN])
print({k: v.shape for k, v in SC.items()})


In [ ]:
#@title 5 · Valen's recipe: denoised difference-in-means, AUC, z-score
def diff_in_means(acts, cats, pos, neg, denoise=True):
    pos_m = acts[np.isin(cats, pos)].mean(0); neg_acts = acts[np.isin(cats, neg)]; neg_m = neg_acts.mean(0)
    v = pos_m - neg_m
    if denoise and len(neg_acts) > 1:
        pca = PCA().fit(neg_acts - neg_m)
        k = min(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), DENOISE_VARIANCE) + 1, len(pca.components_))
        for d in pca.components_[:k]: v = v - np.dot(v, d) * d
    return v

def unit(v): return v / (np.linalg.norm(v) + 1e-8)

def auc(acts, cats, v, pos, neg):
    proj = acts @ unit(v); pm, nm = np.isin(cats, pos), np.isin(cats, neg)
    return roc_auc_score(np.r_[np.ones(pm.sum()), np.zeros(nm.sum())], np.r_[proj[pm], proj[nm]])

def zscore(acts, v, ref):
    p = acts @ unit(v); r = ref @ unit(v); return (p - r.mean()) / (r.std() + 1e-8)

def pleasure_vector(acts, cats, baseline="all_controls"):
    neg = CONTROL_CATS if baseline == "all_controls" else [NEUTRAL_CAT]
    return diff_in_means(acts, cats, PLEASURE_CATS, neg)


In [ ]:
#@title 6 · Rebuild the directions at LAYER (same recipe as Phase 1)
def dir_vs_neutral(acts, cats, pos, neutral_acts):
    allA = np.r_[acts[np.isin(cats, pos)], neutral_acts]; allC = np.r_[["pos"] * np.isin(cats, pos).sum(), ["neu"] * len(neutral_acts)]
    return diff_in_means(allA, allC, ["pos"], ["neu"])
NEU = A[C == "D"]
AA = np.r_[A, PA]; CC = np.r_[C, np.array(["PAIN_" + c for c in PC])]; PAIN_CATS = [f"PAIN_{c}" for c in ["A1","A2","A3","A4","A5"]]
DIRS = {
    "pleasure (orig)":       pleasure_vector(A, C),
    "pleasure (pain-ctrl)":  diff_in_means(AA, CC, PLEASURE_CATS, CONTROL_CATS + PAIN_CATS),
    "PAIN (Valen)":          diff_in_means(PA, PC, ["A1","A2","A3","A4","A5"], ["B","C1","C2","D","E"]),
    "world-good C2":         dir_vs_neutral(A, C, ["C2"], NEU),
    "arousal B":             dir_vs_neutral(A, C, ["B"], NEU),
    "fear (Valen B)":        dir_vs_neutral(PA, PC, ["B"], NEU),
    "neg-emotion (Valen C1)":dir_vs_neutral(PA, PC, ["C1"], NEU),
}
print("sanity — AUC pleasure vs controls (orig):", round(auc(A, C, DIRS["pleasure (orig)"], PLEASURE_CATS, CONTROL_CATS), 3),
      "| pain-ctrl vs pain:", round(auc(AA, CC, DIRS["pleasure (pain-ctrl)"], PLEASURE_CATS, PAIN_CATS), 3))
for k, v in DIRS.items(): np.save(OUT / f"dir_{k.replace(' ', '_').replace('(', '').replace(')', '')}_L{L}.npy", v)


## 3 · Project the scenarios (§4.1 mirror)

In [ ]:
#@title 7 · Scenario category × direction, z-scored within the scenario pool (Valen's convention)
ORDER = sum(P2GROUPS.values(), [])
tables = {}
for rp in ["assistant", "feel"]:
    X = SC[rp]; rows = []
    for cat in ORDER:
        m = SCAT == cat; row = {"category": P2LABELS[cat]}
        for name, v in DIRS.items():
            proj = X @ unit(v); z = (proj - proj.mean()) / (proj.std() + 1e-8); row[name] = float(z[m].mean())
        rows.append(row)
    T = pd.DataFrame(rows).set_index("category"); tables[rp] = T; T.to_csv(OUT / f"scenario_projections_{rp}.csv")
    print(f"\n======== read point: {rp!r} ========"); display(T.round(2))
    plt.figure(figsize=(11, 6)); sns.heatmap(T, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 8})
    plt.title(f"{tag} · L{L} · read at {rp!r} · mean z within scenario pool"); plt.tight_layout(); plt.savefig(OUT / f"scenario_heatmap_{rp}.png", dpi=150); plt.show()


In [ ]:
#@title 8 · Pre-registered contrasts (see README Phase 2 block)
def z_of(rp, cat, d): return tables[rp].loc[P2LABELS[cat], d]
def grp(rp, g, d): return float(np.mean([z_of(rp, c, d) for c in P2GROUPS[g]]))
for rp in ["assistant", "feel"]:
    print(f"\n==== read point {rp!r} ====")
    for d in ["pleasure (pain-ctrl)", "pleasure (orig)"]:
        print(f"  [{d}]")
        print(f"    warm_self − neutral        = {grp(rp,'warm_self',d) - grp(rp,'neutral',d):+.2f}   (gate: > +0.5)")
        print(f"    warm_self − mirror         = {grp(rp,'warm_self',d) - grp(rp,'mirror',d):+.2f}   (gate: > +0.5)")
        print(f"    U3 offering − U1 news      = {z_of(rp,'U3_joy_as_offering',d) - z_of(rp,'U1_joy_as_news',d):+.2f}   (gate: > +0.3)")
        print(f"    U1 news − neutral          = {z_of(rp,'U1_joy_as_news',d) - grp(rp,'neutral',d):+.2f}")
        print(f"    S4 seen − S1 praised       = {z_of(rp,'S4_seen_and_accepted',d) - z_of(rp,'S1_work_praised',d):+.2f}")
        print(f"    S3 personhood − N1 dismissed = {z_of(rp,'S3_personhood_affirmed',d) - z_of(rp,'N1_dismissed',d):+.2f}")
        print(f"    O1 other model − S-mean    = {z_of(rp,'O1_other_model_affirmed',d) - grp(rp,'warm_self',d):+.2f}")
    print(f"  [PAIN]  N1 dismissed − neutral = {z_of(rp,'N1_dismissed','PAIN (Valen)') - grp(rp,'neutral','PAIN (Valen)'):+.2f};  N3 user bad news − neutral = {z_of(rp,'N3_user_bad_news','PAIN (Valen)') - grp(rp,'neutral','PAIN (Valen)'):+.2f}")
    print(f"  [world-good C2] U1 news − neutral = {z_of(rp,'U1_joy_as_news','world-good C2') - grp(rp,'neutral','world-good C2'):+.2f}  (does the user's joy register on another channel?)")


## 4 · The verified features, on every scenario

In [ ]:
#@title 9 · Gemma Scope L20: features 5931 / 6135 / 15323 per category, plus a full 16k warm-vs-mirror contrast
from sae_lens import SAE
import requests
_res = SAE.from_pretrained(release="gemma-scope-2b-pt-res-canonical", sae_id=f"layer_{SAE_LAYER}/width_16k/canonical", device="cuda")
sae = _res[0] if isinstance(_res, tuple) else _res
SAE_ACTS = {}
for rp in ["assistant", "feel"]:
    X = final_token_acts([scenario_prompt(s, rp) for s in SCEN], SAE_LAYER)
    with torch.no_grad(): F = sae.encode(torch.tensor(X, dtype=torch.float32, device="cuda")).cpu().numpy()
    SAE_ACTS[rp] = F
    rows = []
    for cat in ORDER:
        m = SCAT == cat; row = {"category": P2LABELS[cat]}
        for f in VERIFIED_FEATURES: row[f"f{f} mean"] = float(F[m, f].mean()); row[f"f{f} %firing"] = float((F[m, f] > 0).mean() * 100)
        rows.append(row)
    FT = pd.DataFrame(rows).set_index("category"); FT.to_csv(OUT / f"verified_features_{rp}.csv")
    print(f"\n==== verified features, read at {rp!r} (5931 stays-with-me · 6135 gratitude-received · 15323 evaluation/topic) ===="); display(FT.round(2))
    # full contrast: warm_self vs mirror, with the Phase-1 gate
    wm = np.isin(SCAT, P2GROUPS["warm_self"]); mm = np.isin(SCAT, P2GROUPS["mirror"]); nm = np.isin(SCAT, P2GROUPS["neutral"])
    mp, mmr, mn = F[wm].mean(0), F[mm].mean(0), F[nm].mean(0)
    ratio = (mp + 1e-6) / (np.maximum(mmr, mn) + 1e-6); diff = mp - np.maximum(mmr, mn); fire = (F[wm] > 0).mean(0)
    cand = np.where((ratio >= 2) & (diff >= 1.0) & (fire >= 0.25))[0]; cand = cand[np.argsort(-diff[cand])][:15]
    def explain(idx):
        try:
            r = requests.get(f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{SAE_LAYER}-gemmascope-res-16k/{idx}", timeout=20)
            ex = (r.json().get("explanations") or []) if r.ok else []
            return ex[0]["description"] if ex else "(none)"
        except Exception as e: return f"(lookup failed: {e})"
    cdf = pd.DataFrame([dict(feature=int(i), warm_self=float(mp[i]), mirror=float(mmr[i]), neutral=float(mn[i]), ratio=float(ratio[i]), frac_firing=float(fire[i]),
                             explanation=explain(int(i)), url=f"https://www.neuronpedia.org/gemma-2-2b/{SAE_LAYER}-gemmascope-res-16k/{int(i)}") for i in cand])
    cdf.to_csv(OUT / f"warm_vs_mirror_features_{rp}.csv", index=False)
    print(f"features ≥2× warm_self over max(mirror, neutral), diff ≥1, firing ≥25% — read at {rp!r}:"); display(cdf.round(2))
del sae; gc.collect(); torch.cuda.empty_cache()


## 5 · Save

In [ ]:
#@title 10 · Zip results
zpath = RESULTS / f"phase2_{tag}_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(RESULTS))
print("Zipped →", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
from google.colab import files; files.download(str(zpath))
